In [5]:
"""
NBA Game Prediction - Feature Selection Pipeline
==================================================

This module handles feature selection for NBA game prediction models.
Key features:
- Proper temporal train/val/test splitting (COVID seasons excluded)
- Rank-based fusion of ANOVA and MI scores
- Post-selection correlation pruning
- Feature count optimization
- Time-series cross-validation for stability analysis
- Comprehensive metadata output

Usage:
    from feature_selection_pipeline import FeatureSelector
    
    selector = FeatureSelector(engine)
    results = selector.run_full_pipeline()
    selector.save_results(results, 'feature_selection_results.json')
"""

import copy
import json
import warnings
from datetime import datetime

import numpy as np
import pandas as pd
from sklearn.feature_selection import f_classif, mutual_info_classif
from sklearn.metrics import brier_score_loss
from sqlalchemy import create_engine, text
from xgboost import XGBClassifier  # <--- This is the missing import causing your error

warnings.filterwarnings("ignore")

TEAM_LOCATIONS = {
    1610612737: (33.7573, -84.3963),  # ATL
    1610612738: (42.3662, -71.0621),  # BOS
    1610612751: (40.6826, -73.9754),  # BKN
    1610612766: (35.2251, -80.8392),  # CHA
    1610612741: (41.8807, -87.6742),  # CHI
    1610612739: (41.4965, -81.6881),  # CLE
    1610612742: (32.7905, -96.8103),  # DAL
    1610612743: (39.7487, -105.0076),  # DEN
    1610612765: (42.3411, -83.0553),  # DET
    1610612744: (37.7680, -122.3877),  # GSW
    1610612745: (29.7508, -95.3621),  # HOU
    1610612754: (39.7640, -86.1555),  # IND
    1610612746: (33.9425, -118.4081),  # LAC
    1610612747: (34.0430, -118.2673),  # LAL
    1610612763: (35.1382, -90.0505),  # MEM
    1610612748: (25.7814, -80.1870),  # MIA
    1610612749: (43.0451, -87.9172),  # MIL
    1610612750: (44.9795, -93.2761),  # MIN
    1610612740: (29.9490, -90.0821),  # NOP
    1610612752: (40.7505, -73.9934),  # NYK
    1610612760: (35.4634, -97.5151),  # OKC
    1610612753: (28.5392, -81.3839),  # ORL
    1610612755: (39.9012, -75.1720),  # PHI
    1610612756: (33.4457, -112.0712),  # PHX
    1610612757: (45.5316, -122.6668),  # POR
    1610612758: (38.5802, -121.4997),  # SAC
    1610612759: (29.4270, -98.4375),  # SAS
    1610612761: (43.6435, -79.3791),  # TOR
    1610612762: (40.7683, -111.9011),  # UTA
    1610612764: (38.8982, -77.0209),  # WAS
}


class FeatureSelector:
    """
    Comprehensive feature selection for NBA game prediction.

    Implements rank-based fusion, correlation pruning, and stability analysis.
    """

    def __init__(self, engine, random_state=42):
        """
        Initialize feature selector.

        Args:
            engine: SQLAlchemy engine for database connection
            random_state: Random seed for reproducibility
        """
        self.engine = engine
        self.random_state = random_state
        np.random.seed(random_state)

        # Consistent IDs
        self.train_seasons = [f"2{year}" for year in range(2009, 2019)]
        self.val_seasons = ["22021"]
        self.test_seasons = ["22022", "22023", "22024"]
        self.excluded_seasons = ["22019", "22020"]

        print("=" * 80)
        print("NBA FEATURE SELECTION PIPELINE")
        print("=" * 80)
        print("\n📊 Season Configuration:")
        print(f"  Training:   {', '.join(self.train_seasons)} ({len(self.train_seasons)} seasons)")
        print(f"  Validation: {', '.join(self.val_seasons)} ({len(self.val_seasons)} season)")
        print(f"  Test:       {', '.join(self.test_seasons)} ({len(self.test_seasons)} seasons)")
        print(f"  Excluded:   {', '.join(self.excluded_seasons)} (COVID impact)")
        print()

    def load_data(self):
        """
        Load team and player data with proper season filtering.

        Returns:
            dict: Contains train, val, test DataFrames
        """
        print("🔄 Loading data from database...")
        team_df = self._get_team_data()
        player_df = self._get_player_data()

        # 1. Merge and Create features (Keeping IDs for now)
        ml_df = self._merge_and_create_features(team_df, player_df)

        # 2. Add Advanced 4 Factors
        adv_df = self._get_advanced_features()

        # Merge Home Team (team1)
        # We rename columns to avoid collision and clarify ownership
        home_adv = adv_df.rename(
            columns={
                "team_id": "team1_id",
                "avg_efg_pct": "team_avg_efg_pct",
                "avg_tov_pct": "team_avg_tov_pct",
                "avg_orb_pct": "team_avg_orb_pct",
                "avg_ft_rate": "team_avg_ft_rate",
            }
        )
        ml_df = ml_df.merge(home_adv, on=["game_id", "team1_id"], how="left")

        # Merge Away Team (team2)
        opp_adv = adv_df.rename(
            columns={
                "team_id": "team2_id",
                "avg_efg_pct": "opp_avg_efg_pct",
                "avg_tov_pct": "opp_avg_tov_pct",
                "avg_orb_pct": "opp_avg_orb_pct",
                "avg_ft_rate": "opp_avg_ft_rate",
            }
        )
        ml_df = ml_df.merge(opp_adv, on=["game_id", "team2_id"], how="left")

        # Calculate Differentials
        features = ["avg_efg_pct", "avg_tov_pct", "avg_orb_pct", "avg_ft_rate"]
        for f in features:
            ml_df[f"diff_{f}"] = ml_df[f"team_{f}"] - ml_df[f"opp_{f}"]
        # --- NEW CODE END ---

        ml_df = self._calculate_sos(ml_df)
        ml_df = ml_df.fillna(0)

        # 3. Add target variable
        target_df = self._get_target_variable()
        ml_df = ml_df.merge(target_df, on="game_id", how="inner")

        print(f"  ✓ Created ML dataset: {ml_df.shape[0]:,} rows")

        # 4. Split by season (The split function now handles the final column drops)
        data_splits = self._split_by_season(ml_df)

        print("\n📈 Data Split Summary:")
        print(f"  Training:   {data_splits['train'].shape[0]:,} games")
        print(f"  Validation: {data_splits['val'].shape[0]:,} games")
        print(f"  Test:       {data_splits['test'].shape[0]:,} games")
        print(f"  Total:      {sum(df.shape[0] for df in data_splits.values()):,} games")

        return data_splits

    def _get_advanced_features(self):
        """
        Fetches the 4 Factors calculated from rolling averages directly from the DB.
        Returns a DataFrame indexed by (game_id, team_id).
        """
        print("  📊 Fetching advanced 4 Factors (League Quality)...")
        with self.engine.connect() as conn:
            query = text("""
            SELECT 
                base.game_id,
                base.team_id,
                
                -- 1. EFFECTIVE FIELD GOAL %
                (hero.avg_team_fgm + 0.5 * hero.avg_team_fg3m) / NULLIF(hero.avg_team_fga, 0) as avg_efg_pct,

                -- 2. TURNOVER %
                hero.avg_team_tov / NULLIF(
                    (hero.avg_team_fga + 0.44 * hero.avg_team_fta - hero.avg_team_oreb + hero.avg_team_tov), 0
                ) as avg_tov_pct,

                -- 3. OFFENSIVE REBOUND %
                hero.avg_team_oreb / NULLIF(
                    (hero.avg_team_oreb + villain.avg_team_dreb), 0
                ) as avg_orb_pct,

                -- 4. FREE THROW RATE 
                hero.avg_team_ftm / NULLIF(hero.avg_team_fga, 0) as avg_ft_rate

            FROM team_game_stats base
            JOIN team_average_game_stats hero 
                ON base.game_id = hero.game_id AND base.team_id = hero.team_id
            JOIN team_average_game_stats villain 
                ON base.game_id = villain.game_id AND base.opponent_id = villain.team_id
            """)

            return pd.read_sql(query, conn)

    def _calculate_sos(self, df):
        """
        Calculates cumulative Strength of Schedule (SOS) for both teams.
        Returns the original DataFrame with 'diff_sos' added.
        """
        print("  📊 Calculating Strength of Schedule (SOS)...")

        # 1. Create a "Long" view (Stack Home and Away games)
        # We need a single list of [Team, Date, OpponentStrength]
        home_view = df[["game_id", "game_date", "team1_id", "opp_plus_minus"]].rename(
            columns={"team1_id": "team_id", "opp_plus_minus": "opp_strength"}
        )
        away_view = df[["game_id", "game_date", "team2_id", "team_plus_minus"]].rename(
            columns={"team2_id": "team_id", "team_plus_minus": "opp_strength"}
        )

        schedule = pd.concat([home_view, away_view])

        # 2. Sort Chronologically by Team
        schedule["game_date"] = pd.to_datetime(schedule["game_date"])
        schedule = schedule.sort_values(["team_id", "game_date"])

        # 3. Calculate Expanding Mean (Shift 1 to exclude current game)
        schedule["sos"] = schedule.groupby("team_id")["opp_strength"].transform(lambda x: x.shift(1).expanding().mean())
        schedule["sos"] = schedule["sos"].fillna(0)

        # 4. Map back to Main DataFrame
        # Map Home Team's SOS
        df = df.merge(
            schedule[["game_id", "team_id", "sos"]].rename(columns={"team_id": "team1_id", "sos": "team1_sos"}),
            on=["game_id", "team1_id"],
            how="left",
        )
        # Map Away Team's SOS
        df = df.merge(
            schedule[["game_id", "team_id", "sos"]].rename(columns={"team_id": "team2_id", "sos": "team2_sos"}),
            on=["game_id", "team2_id"],
            how="left",
        )

        # 5. Create the Feature (Home SOS - Away SOS)
        df["diff_sos"] = df["team1_sos"] - df["team2_sos"]

        return df

    def _haversine(self, lat1, lon1, lat2, lon2):
        """Vectorized Haversine distance calculation."""
        lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
        dlon = lon2 - lon1
        dlat = lat2 - lat1
        a = np.sin(dlat / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
        c = 2 * np.arcsin(np.sqrt(a))
        return c * 3956  # Miles

    def _get_travel_and_rest_data(self):
        """Fetches minimal schedule data to calculate rest and travel features."""
        with self.engine.connect() as conn:
            query = text("""
                SELECT 
                    team_id, game_id, team_game_date, team_matchup,
                    CASE WHEN team_matchup LIKE '%vs.%' THEN 1 ELSE 0 END as is_home
                FROM team_game_stats
                ORDER BY team_id, team_game_date
            """)
            df = pd.read_sql(query, conn)

        # 1. Setup Coordinates
        df["game_date"] = pd.to_datetime(df["team_game_date"])

        team_lat = {k: v[0] for k, v in TEAM_LOCATIONS.items()}
        team_lon = {k: v[1] for k, v in TEAM_LOCATIONS.items()}

        df["my_lat"] = df["team_id"].map(team_lat)
        df["my_lon"] = df["team_id"].map(team_lon)

        # Self-join to get opponent info
        df_opp = df[["game_id", "team_id", "my_lat", "my_lon"]].rename(
            columns={"team_id": "opp_id", "my_lat": "opp_lat", "my_lon": "opp_lon"}
        )
        df = df.merge(df_opp, on="game_id")
        df = df[df["team_id"] != df["opp_id"]].copy()

        # Determine Game Location
        df["loc_lat"] = np.where(df["is_home"] == 1, df["my_lat"], df["opp_lat"])
        df["loc_lon"] = np.where(df["is_home"] == 1, df["my_lon"], df["opp_lon"])

        # 2. Sort by Team sequence
        df = df.sort_values(["team_id", "game_date"])

        # 3. Calculate Lags
        df["prev_date"] = df.groupby("team_id")["game_date"].shift(1)
        df["prev_lat"] = df.groupby("team_id")["loc_lat"].shift(1)
        df["prev_lon"] = df.groupby("team_id")["loc_lon"].shift(1)

        # Fill First Game of Season
        df["prev_date"] = df["prev_date"].fillna(df["game_date"] - pd.Timedelta(days=7))
        df["prev_lat"] = df["prev_lat"].fillna(df["my_lat"])
        df["prev_lon"] = df["prev_lon"].fillna(df["my_lon"])

        # 4. Compute Metrics
        df["rest_days"] = (df["game_date"] - df["prev_date"]).dt.days
        df["rest_days"] = df["rest_days"].clip(0, 7)

        df["travel_dist"] = self._haversine(df["prev_lat"], df["prev_lon"], df["loc_lat"], df["loc_lon"])

        return df.set_index(["game_id", "team_id"])[["rest_days", "travel_dist"]]

    def _get_team_data(self):
        """Load team statistics from database."""
        with self.engine.connect() as conn:
            query = text("""
                WITH game_teams AS (
                    SELECT DISTINCT
                        tgs.game_id,
                        tgs.season_id,
                        tgs.team_game_date as game_date,
                        -- Home team = team1 (matchup contains 'vs.')
                        MAX(CASE WHEN tgs.team_matchup LIKE '%vs.%' THEN tgs.team_id END) as team1_id,
                        -- Away team = team2 (matchup contains '@')
                        MAX(CASE WHEN tgs.team_matchup LIKE '%@%' THEN tgs.team_id END) as team2_id
                    FROM team_game_stats tgs
                    WHERE tgs.season_id NOT IN :excluded_seasons
                    GROUP BY tgs.game_id, tgs.season_id, tgs.team_game_date
                )
                SELECT 
                    gt.game_id,
                    gt.season_id,
                    gt.game_date,
                    gt.team1_id,
                    gt.team2_id,
                    
                    -- Team 1 (HOME) stats
                    t1.avg_team_plus_minus as team_plus_minus,
                    t1.avg_team_pts as team_pts,
                    t1.avg_team_fgm as team_fgm,
                    t1.avg_team_fga as team_fga,
                    t1.avg_team_fg_pct as team_fg_pct,
                    t1.avg_team_fg3m as team_fg3m,
                    t1.avg_team_fg3a as team_fg3a,
                    t1.avg_team_fg3_pct as team_fg3_pct,
                    t1.avg_team_ftm as team_ftm,
                    t1.avg_team_fta as team_fta,
                    t1.avg_team_ft_pct as team_ft_pct,
                    t1.avg_team_oreb as team_oreb,
                    t1.avg_team_dreb as team_dreb,
                    t1.avg_team_reb as team_reb,
                    t1.avg_team_ast as team_ast,
                    t1.avg_team_stl as team_stl,
                    t1.avg_team_blk as team_blk,
                    t1.avg_team_tov as team_tov,
                    t1.avg_team_pf as team_pf,
                    t1.games_in_average as team_games_in_avg,
                    
                    -- Team 2 (AWAY) stats
                    t2.avg_team_plus_minus as opp_plus_minus,
                    t2.avg_team_pts as opp_pts,
                    t2.avg_team_fgm as opp_fgm,
                    t2.avg_team_fga as opp_fga,
                    t2.avg_team_fg_pct as opp_fg_pct,
                    t2.avg_team_fg3m as opp_fg3m,
                    t2.avg_team_fg3a as opp_fg3a,
                    t2.avg_team_fg3_pct as opp_fg3_pct,
                    t2.avg_team_ftm as opp_ftm,
                    t2.avg_team_fta as opp_fta,
                    t2.avg_team_ft_pct as opp_ft_pct,
                    t2.avg_team_oreb as opp_oreb,
                    t2.avg_team_dreb as opp_dreb,
                    t2.avg_team_reb as opp_reb,
                    t2.avg_team_ast as opp_ast,
                    t2.avg_team_stl as opp_stl,
                    t2.avg_team_blk as opp_blk,
                    t2.avg_team_tov as opp_tov,
                    t2.avg_team_pf as opp_pf,
                    t2.games_in_average as opp_games_in_avg,
                    
                    -- Differences (HOME - AWAY)
                    t1.avg_team_pts - t2.avg_team_pts as diff_pts,
                    t1.avg_team_fgm - t2.avg_team_fgm as diff_fgm,
                    t1.avg_team_fga - t2.avg_team_fga as diff_fga,
                    t1.avg_team_fg_pct - t2.avg_team_fg_pct as diff_fg_pct,
                    t1.avg_team_fg3m - t2.avg_team_fg3m as diff_fg3m,
                    t1.avg_team_fg3a - t2.avg_team_fg3a as diff_fg3a,
                    t1.avg_team_fg3_pct - t2.avg_team_fg3_pct as diff_fg3_pct,
                    t1.avg_team_ftm - t2.avg_team_ftm as diff_ftm,
                    t1.avg_team_fta - t2.avg_team_fta as diff_fta,
                    t1.avg_team_ft_pct - t2.avg_team_ft_pct as diff_ft_pct,
                    t1.avg_team_oreb - t2.avg_team_oreb as diff_oreb,
                    t1.avg_team_dreb - t2.avg_team_dreb as diff_dreb,
                    t1.avg_team_reb - t2.avg_team_reb as diff_reb,
                    t1.avg_team_ast - t2.avg_team_ast as diff_ast,
                    t1.avg_team_stl - t2.avg_team_stl as diff_stl,
                    t1.avg_team_blk - t2.avg_team_blk as diff_blk,
                    t1.avg_team_tov - t2.avg_team_tov as diff_tov,
                    t1.avg_team_pf - t2.avg_team_pf as diff_pf
                    
                FROM game_teams gt
                JOIN team_average_game_stats t1 ON gt.game_id = t1.game_id AND gt.team1_id = t1.team_id
                JOIN team_average_game_stats t2 ON gt.game_id = t2.game_id AND gt.team2_id = t2.team_id
                WHERE t1.games_in_average >= 5
                  AND t2.games_in_average >= 5
                ORDER BY gt.game_date, gt.game_id;
            """)

            # return pd.read_sql(
            #     query,
            #     conn,
            #     params={'excluded_seasons': tuple(self.excluded_seasons)}
            # )
            main_df = pd.read_sql(query, conn, params={"excluded_seasons": tuple(self.excluded_seasons)})

            # Calculate Travel/Rest
            print("  ✈️  Calculating travel logistics...")
            travel_lookup = self._get_travel_and_rest_data()

            # Merge for Team 1 (Home)
            main_df = main_df.join(
                travel_lookup.rename(columns={"rest_days": "team_rest", "travel_dist": "team_dist"}),
                on=["game_id", "team1_id"],
            )

            # Merge for Team 2 (Away)
            main_df = main_df.join(
                travel_lookup.rename(columns={"rest_days": "opp_rest", "travel_dist": "opp_dist"}),
                on=["game_id", "team2_id"],
            )

            # Create Differential Features
            main_df["diff_rest"] = main_df["team_rest"] - main_df["opp_rest"]
            main_df["diff_dist"] = main_df["team_dist"] - main_df["opp_dist"]

            return main_df.fillna(0)

    def _get_player_data(self):
        """Load player statistics from database."""
        with self.engine.connect() as conn:
            query = text("""
                WITH game_teams AS (
                    SELECT DISTINCT
                        tgs.game_id,
                        tgs.season_id,
                        MAX(CASE WHEN tgs.team_matchup LIKE '%vs.%' THEN tgs.team_id END) as team1_id,
                        MAX(CASE WHEN tgs.team_matchup LIKE '%@%' THEN tgs.team_id END) as team2_id
                    FROM team_game_stats tgs
                    WHERE tgs.season_id NOT IN :excluded_seasons
                    GROUP BY tgs.game_id, tgs.season_id
                ),
                ranked_players AS (
                    SELECT 
                        pgs.game_id,
                        pgs.season_id,
                        pgs.team_id,
                        pgs.player_id,
                        p.player_name,
                        pgs.avg_min,
                        pgs.avg_pts,
                        pgs.avg_fgm,
                        pgs.avg_fga,
                        pgs.avg_fg_pct,
                        pgs.avg_fg3m,
                        pgs.avg_fg3a,
                        pgs.avg_fg3_pct,
                        pgs.avg_ftm,
                        pgs.avg_fta,
                        pgs.avg_ft_pct,
                        pgs.avg_oreb,
                        pgs.avg_dreb,
                        pgs.avg_reb,
                        pgs.avg_ast,
                        pgs.avg_stl,
                        pgs.avg_blk,
                        pgs.avg_tov,
                        pgs.avg_pf,
                        pgs.avg_plus_minus,
                        pgs.games_in_average,
                        ROW_NUMBER() OVER (PARTITION BY pgs.game_id, pgs.team_id ORDER BY pgs.avg_min DESC) as player_rank
                    FROM player_average_game_stats pgs
                    JOIN players p ON pgs.player_id = p.player_id
                    WHERE pgs.avg_min > 0
                )
                -- HOME team players
                SELECT 
                    gt.game_id,
                    gt.season_id,
                    'team' as player_team_type,
                    gt.team1_id as team_id,
                    rp.player_rank,
                    rp.player_id,
                    rp.player_name,
                    rp.avg_min,
                    rp.avg_pts,
                    rp.avg_fgm,
                    rp.avg_fga,
                    rp.avg_fg_pct,
                    rp.avg_fg3m,
                    rp.avg_fg3a,
                    rp.avg_fg3_pct,
                    rp.avg_ftm,
                    rp.avg_fta,
                    rp.avg_ft_pct,
                    rp.avg_oreb,
                    rp.avg_dreb,
                    rp.avg_reb,
                    rp.avg_ast,
                    rp.avg_stl,
                    rp.avg_blk,
                    rp.avg_tov,
                    rp.avg_pf,
                    rp.avg_plus_minus,
                    rp.games_in_average
                FROM game_teams gt
                JOIN ranked_players rp ON gt.game_id = rp.game_id AND gt.team1_id = rp.team_id
                WHERE rp.player_rank <= 10
                
                UNION ALL
                
                -- AWAY team players
                SELECT 
                    gt.game_id,
                    gt.season_id,
                    'opp' as player_team_type,
                    gt.team2_id as team_id,
                    rp.player_rank,
                    rp.player_id,
                    rp.player_name,
                    rp.avg_min,
                    rp.avg_pts,
                    rp.avg_fgm,
                    rp.avg_fga,
                    rp.avg_fg_pct,
                    rp.avg_fg3m,
                    rp.avg_fg3a,
                    rp.avg_fg3_pct,
                    rp.avg_ftm,
                    rp.avg_fta,
                    rp.avg_ft_pct,
                    rp.avg_oreb,
                    rp.avg_dreb,
                    rp.avg_reb,
                    rp.avg_ast,
                    rp.avg_stl,
                    rp.avg_blk,
                    rp.avg_tov,
                    rp.avg_pf,
                    rp.avg_plus_minus,
                    rp.games_in_average
                FROM game_teams gt
                JOIN ranked_players rp ON gt.game_id = rp.game_id AND gt.team2_id = rp.team_id
                WHERE rp.player_rank <= 10
                
                ORDER BY game_id, player_team_type, player_rank;
            """)

            return pd.read_sql(query, conn, params={"excluded_seasons": tuple(self.excluded_seasons)})

    def _merge_and_create_features(self, team_df, player_df):
        """
        Merge team and player data, pivot player stats.

        Args:
            team_df: Team statistics DataFrame
            player_df: Player statistics DataFrame

        Returns:
            DataFrame: ML-ready dataset
        """
        player_stats_cols = [
            "avg_min",
            "avg_pts",
            "avg_fgm",
            "avg_fga",
            "avg_fg_pct",
            "avg_fg3m",
            "avg_fg3a",
            "avg_fg3_pct",
            "avg_ftm",
            "avg_fta",
            "avg_ft_pct",
            "avg_oreb",
            "avg_dreb",
            "avg_reb",
            "avg_ast",
            "avg_stl",
            "avg_blk",
            "avg_tov",
            "avg_pf",
            "avg_plus_minus",
        ]

        # Efficient pivoting
        pivot = player_df.pivot_table(
            index="game_id",
            columns=["player_team_type", "player_rank"],
            values=player_stats_cols,
            aggfunc="first",
        )
        # Flatten multi-index columns: 'avg_pts', 'team', 1 -> 'team_pllayer1_avg_pts'
        pivot.columns = [f"{tt}_player{r}_{s}" for s, tt, r in pivot.columns]

        return team_df.merge(pivot, on="game_id", how="left")

    def _get_target_variable(self):  # Removed 'df' argument
        """Fetches final scores to create labels."""
        with self.engine.connect() as conn:
            query = text("""
                SELECT game_id,
                       CASE WHEN team_matchup LIKE '%vs.%' AND team_wl = 'W' THEN 1
                            WHEN team_matchup LIKE '%@%' AND team_wl = 'L' THEN 1
                            ELSE 0 END as target_win
                FROM team_game_stats
                WHERE season_id NOT IN :excluded_seasons
            """)
            target_df = pd.read_sql(query, conn, params={"excluded_seasons": tuple(self.excluded_seasons)})

            # We drop duplicates to ensure we have exactly one label per game_id
            return target_df.drop_duplicates("game_id")

    def _split_by_season(self, ml_df):
        """Final stage: Splits data and removes administrative columns."""
        splits = {
            "train": ml_df[ml_df["season_id"].isin(self.train_seasons)].copy(),
            "val": ml_df[ml_df["season_id"].isin(self.val_seasons)].copy(),
            "test": ml_df[ml_df["season_id"].isin(self.test_seasons)].copy(),
        }

        drop_cols = ["game_id", "season_id", "game_date", "team1_id", "team2_id"]

        for key in splits:
            # We use errors='ignore' in case some columns were already missing
            splits[key] = splits[key].drop(columns=drop_cols, errors="ignore")

        return splits

    def select_features(self, X_train, y_train, correlation_threshold=0.85):
        """
        Run feature selection: ANOVA + MI with rank-based fusion and correlation pruning.

        Args:
            X_train: Training features
            y_train: Training target
            correlation_threshold: Correlation cutoff for pruning

        Returns:
            dict: Feature selection results with rankings and scores
        """
        print("\n" + "=" * 80)
        print("FEATURE SELECTION")
        print("=" * 80)

        feature_names = X_train.columns.tolist()
        print(f"\n📊 Starting with {len(feature_names)} features")

        # Remove zero/low variance features
        print("\n🔍 Step 1: Variance filtering...")
        variance = X_train.var()
        low_var_features = variance[variance < 1e-8].index.tolist()
        if low_var_features:
            print(f"  ✗ Removing {len(low_var_features)} near-zero variance features")
            X_train = X_train.drop(columns=low_var_features)
            feature_names = X_train.columns.tolist()
        else:
            print("  ✓ All features have sufficient variance")

        # ANOVA F-test
        print("\n🔍 Step 2: ANOVA F-test...")
        anova_scores, _ = f_classif(X_train, y_train)
        anova_scores = pd.Series(anova_scores, index=feature_names)
        anova_ranks = anova_scores.rank(ascending=False, method="min")
        print(f"  ✓ Computed F-statistics for {len(feature_names)} features")

        # Mutual Information
        print("\n🔍 Step 3: Mutual Information...")
        mi_scores = mutual_info_classif(
            X_train, y_train, discrete_features=False, random_state=self.random_state, n_neighbors=5
        )
        mi_scores = pd.Series(mi_scores, index=feature_names)
        mi_ranks = mi_scores.rank(ascending=False, method="min")
        print(f"  ✓ Computed MI scores for {len(feature_names)} features")

        # Rank-based fusion (using reciprocal rank for higher weight on top features)
        print("\n🔍 Step 4: Rank-based fusion...")
        reciprocal_anova = 1 / anova_ranks
        reciprocal_mi = 1 / mi_ranks
        combined_score = reciprocal_anova + reciprocal_mi
        combined_ranks = combined_score.rank(ascending=False, method="min")

        print("  ✓ Combined rankings using reciprocal rank fusion")

        # Create feature importance dataframe
        feature_importance = pd.DataFrame(
            {
                "feature": feature_names,
                "anova_score": anova_scores.values,
                "anova_rank": anova_ranks.values,
                "mi_score": mi_scores.values,
                "mi_rank": mi_ranks.values,
                "combined_score": combined_score.values,
                "combined_rank": combined_ranks.values,
            }
        ).sort_values("combined_rank")

        # Correlation-based pruning AFTER ranking
        print(f"\n🔍 Step 5: Correlation pruning (threshold={correlation_threshold})...")
        selected_features = self._correlation_pruning(
            X_train, feature_importance["feature"].tolist(), correlation_threshold
        )

        print(f"  ✓ Retained {len(selected_features)} features after correlation pruning")
        print(f"  ✗ Removed {len(feature_names) - len(selected_features)} highly correlated features")

        # Update feature importance to mark pruned features
        feature_importance["selected"] = feature_importance["feature"].isin(selected_features)

        print("\n✅ Feature selection complete!")
        print(f"   Final feature count: {len(selected_features)} / {len(feature_names)}")

        return {
            "feature_importance": feature_importance,
            "selected_features": selected_features,
            "low_variance_features": low_var_features,
            "correlation_threshold": correlation_threshold,
        }

    def _correlation_pruning(self, X, ranked_features, threshold):
        """
        Remove highly correlated features, keeping higher-ranked ones.

        Args:
            X: Feature DataFrame
            ranked_features: List of features in rank order (best first)
            threshold: Correlation threshold for removal

        Returns:
            list: Features to keep after pruning
        """
        selected = []
        removed = []

        for feature in ranked_features:
            if len(selected) == 0:
                selected.append(feature)
                continue

            # Check correlation with already selected features
            correlations = X[selected + [feature]].corr()[feature][:-1].abs()
            max_corr = correlations.max()

            if max_corr < threshold:
                selected.append(feature)
            else:
                # Find which feature it's correlated with
                correlated_with = correlations.idxmax()
                removed.append(
                    {
                        "feature": feature,
                        "correlated_with": correlated_with,
                        "correlation": max_corr,
                    }
                )

        if removed:
            print("\n  📋 Correlation pruning details (showing first 10):")
            for item in removed[:10]:
                print(f"     • {item['feature']} ↔ {item['correlated_with']} (r={item['correlation']:.3f})")
            if len(removed) > 10:
                print(f"     ... and {len(removed) - 10} more")

        return selected

    def time_series_stability_analysis(self, X_train, y_train, selected_features, n_splits=5):
        """
        Test feature selection stability using expanding window time-series CV.

        Args:
            X_train: Training features (must be chronologically ordered)
            y_train: Training target
            selected_features: List of selected features
            n_splits: Number of CV splits

        Returns:
            dict: Stability analysis results
        """
        print("\n" + "=" * 80)
        print("TIME-SERIES STABILITY ANALYSIS")
        print("=" * 80)
        print(f"\n🔄 Running {n_splits}-fold expanding window CV")

        n_samples = len(X_train)
        fold_size = n_samples // (n_splits + 1)

        feature_selection_counts = {}

        for fold in range(n_splits):
            # Expanding window: train on progressively more data
            train_end = (fold + 2) * fold_size
            val_start = train_end
            val_end = min(val_start + fold_size, n_samples)

            X_fold_train = X_train.iloc[:train_end]
            y_fold_train = y_train.iloc[:train_end]

            print(f"\n  Fold {fold + 1}/{n_splits}: Training on {train_end:,} samples")

            # Run feature selection on this fold
            fold_results = self.select_features(X_fold_train, y_fold_train, correlation_threshold=0.70)

            # Count how many times each feature was selected
            for feature in fold_results["selected_features"]:
                feature_selection_counts[feature] = feature_selection_counts.get(feature, 0) + 1

        # Calculate stability metrics
        total_unique_features = len(feature_selection_counts)
        stable_features = [
            f for f, count in feature_selection_counts.items() if count >= n_splits * 0.6
        ]  # Selected in 60%+ of folds

        stability_df = pd.DataFrame(
            [
                {
                    "feature": feature,
                    "selection_frequency": count / n_splits,
                    "times_selected": count,
                    "stable": count >= n_splits * 0.6,
                }
                for feature, count in feature_selection_counts.items()
            ]
        ).sort_values("selection_frequency", ascending=False)

        print("\n✅ Stability analysis complete!")
        print(f"   Total unique features selected: {total_unique_features}")
        print(f"   Stable features (≥60% selection): {len(stable_features)}")
        print(f"   Unstable features: {total_unique_features - len(stable_features)}")

        return {
            "stability_df": stability_df,
            "stable_features": stable_features,
            "n_splits": n_splits,
            "stability_threshold": 0.6,
        }

    def optimize_feature_count(
        self,
        X_train,
        y_train,
        X_val,
        y_val,
        ranked_features,
        coarse_step=10,
        fine_step=2,
        tolerance=0.002,
    ):
        """
        Find the smallest feature count that achieves near-optimal validation performance.

        Uses a two-pass approach:
        1. Coarse search with larger step to find the general region
        2. Fine search around the best region for precision

        Args:
            X_train, y_train: Training data
            X_val, y_val: Validation data
            ranked_features: List of feature names, ordered by importance
            coarse_step: Step size for initial search (default: 10)
            fine_step: Step size for refined search (default: 2)
            tolerance: Accept smallest N within this Brier delta of best (default: 0.002)

        Returns:
            dict: optimal_n_features, optimal_brier_score, all_results
        """
        print("\n" + "=" * 80)
        print("OPTIMIZING FEATURE COUNT")
        print(f"  Coarse step: {coarse_step} | Fine step: {fine_step} | Tolerance: {tolerance}")
        print("=" * 80)

        all_results = []

        def evaluate_n_features(n):
            """Train a lightweight model and return validation Brier score."""
            current_features = ranked_features[:n]

            model = XGBClassifier(
                n_estimators=100,
                max_depth=4,
                learning_rate=0.1,
                random_state=self.random_state,
                eval_metric="logloss",
                verbosity=0,
                early_stopping_rounds=10,
            )

            model.fit(
                X_train[current_features],
                y_train,
                eval_set=[(X_val[current_features], y_val)],
                verbose=False,
            )

            val_probs = model.predict_proba(X_val[current_features])[:, 1]
            brier = brier_score_loss(y_val, val_probs)

            return brier

        # =========================================================================
        # PASS 1: Coarse search
        # =========================================================================
        print(f"\n📊 Pass 1: Coarse search (step={coarse_step})")

        min_features = 10
        max_features = len(ranked_features)
        coarse_range = list(range(min_features, max_features + 1, coarse_step))

        # Ensure we test the maximum
        if coarse_range[-1] != max_features:
            coarse_range.append(max_features)

        coarse_results = []
        for n in coarse_range:
            brier = evaluate_n_features(n)
            coarse_results.append({"n_features": n, "brier_score": brier, "pass": "coarse"})
            print(f"   {n:4d} features | Brier: {brier:.5f}")

        all_results.extend(coarse_results)

        # Find best from coarse pass
        coarse_best = min(coarse_results, key=lambda x: x["brier_score"])
        coarse_best_n = coarse_best["n_features"]
        coarse_best_brier = coarse_best["brier_score"]

        print(f"\n   Coarse best: {coarse_best_n} features (Brier: {coarse_best_brier:.5f})")

        # =========================================================================
        # PASS 2: Fine search around best region
        # =========================================================================
        print(f"\n🔬 Pass 2: Fine search around {coarse_best_n} (step={fine_step})")

        # Search window: ±coarse_step around the best
        fine_min = max(min_features, coarse_best_n - coarse_step)
        fine_max = min(max_features, coarse_best_n + coarse_step)
        fine_range = list(range(fine_min, fine_max + 1, fine_step))

        # Skip values we already tested
        tested_ns = {r["n_features"] for r in all_results}
        fine_range = [n for n in fine_range if n not in tested_ns]

        fine_results = []
        for n in fine_range:
            brier = evaluate_n_features(n)
            fine_results.append({"n_features": n, "brier_score": brier, "pass": "fine"})
            print(f"   {n:4d} features | Brier: {brier:.5f}")

        all_results.extend(fine_results)

        # =========================================================================
        # SELECT OPTIMAL: Smallest N within tolerance of best
        # =========================================================================
        best_brier = min(r["brier_score"] for r in all_results)

        # Find all Ns within tolerance, pick the smallest
        candidates = [r for r in all_results if r["brier_score"] <= best_brier + tolerance]
        optimal = min(candidates, key=lambda x: x["n_features"])

        optimal_n = optimal["n_features"]
        optimal_brier = optimal["brier_score"]

        # Sort results for output
        all_results_sorted = sorted(all_results, key=lambda x: x["n_features"])

        print("\n" + "=" * 80)
        print("✅ OPTIMIZATION COMPLETE")
        print(f"   Best Brier Score:      {best_brier:.5f}")
        print(f"   Tolerance:             {tolerance}")
        print(f"   Optimal Feature Count: {optimal_n} (Brier: {optimal_brier:.5f})")

        if optimal_brier > best_brier:
            print(
                f"   → Selected smaller N ({optimal_n}) over absolute best "
                f"(saved {min(r['n_features'] for r in all_results if r['brier_score'] == best_brier) - optimal_n} features)"
            )

        print("=" * 80)

        return {
            "optimal_n_features": optimal_n,
            "optimal_brier_score": optimal_brier,
            "best_brier_score": best_brier,
            "tolerance": tolerance,
            "results": all_results_sorted,
        }

    def run_full_pipeline(self, run_stability=True, stability_threshold=0.6):
        """
        Execute the robust feature selection pipeline.

        Returns:
            dict: Complete data splits and a vetted 'feature_registry'
        """
        start_time = datetime.now()

        # 1. Load and Split (using the fixed temporal logic we discussed)
        data_splits = self.load_data()

        X_train = data_splits["train"].drop(columns=["target_win"])
        y_train = data_splits["train"]["target_win"]
        X_val = data_splits["val"].drop(columns=["target_win"])
        y_val = data_splits["val"]["target_win"]

        # 2. Global Ranking & Pruning
        # This identifies the strongest features across the entire training history
        selection_results = self.select_features(X_train, y_train)
        master_ranked_list = selection_results["selected_features"]

        # 3. Stability Filtering (Expanding Window Check)
        if run_stability:
            # My version (Clean)
            stability_results = self.time_series_stability_analysis(X_train, y_train, master_ranked_list)
            # The method already did the filtering and kept the sort order:
            final_features = stability_results["stable_features"]

            print(f"\n💎 Stability Analysis: {len(final_features)} features retained.")
        else:
            stability_results = None
            final_features = master_ranked_list

        # 4. OPTIMIZATION
        # Optimize using the vetted features
        optimization_results = self.optimize_feature_count(
            X_train,
            y_train,
            X_val,
            y_val,
            final_features,
            coarse_step=10,
            fine_step=2,
            tolerance=0.001,
        )

        elapsed = (datetime.now() - start_time).total_seconds()

        print("\n" + "=" * 80)
        print("PIPELINE COMPLETE")
        print(f"Total time: {elapsed:.1f} seconds")
        print(f"Optimal N: {optimization_results['optimal_n_features']}")
        print("=" * 80)

        # Construct the EXACT dictionary structure train.ipynb expects
        return {
            "metadata": {
                "timestamp": datetime.now().isoformat(),
                "train_seasons": self.train_seasons,
                "val_seasons": self.val_seasons,
                "test_seasons": self.test_seasons,
                "excluded_seasons": self.excluded_seasons,
                "elapsed_seconds": elapsed,
            },
            "selection": {
                "selected_features": final_features,
                "full_importance_table": selection_results["feature_importance"].to_dict("records"),
            },
            "optimization": optimization_results,
            "stability": stability_results["stability_df"].to_dict("records") if stability_results else None,
        }

    # def save_results(self, results, filepath='feature_selection_results.json'):
    #     """Save results to JSON file."""
    #     print(f"\n💾 Saving results to {filepath}...")
    #     if results.get('stability'):
    #         results['stability']['stability_df'] = results['stability']['stability_df'].to_dict('records')
    #     # Directly save the structured results since run_full_pipeline now
    #     # outputs the correct format
    #     with open(filepath, 'w') as f:
    #         json.dump(results, f, indent=2)

    #     print(f"✅ Results saved successfully!")
    #     return filepath
    def save_results(self, results, filepath="feature_selection_results.json"):
        # === ROBUST SAVE FIX ===
        # Use deepcopy to avoid mutating the original object in memory
        # This prevents "run twice = crash" errors
        results_to_save = copy.deepcopy(results)

        print(f"\n💾 Saving results to {filepath}...")

        # Safe conversion of DataFrame to dict
        if results_to_save.get("stability") and isinstance(results_to_save["stability"], dict):
            if "stability_df" in results_to_save["stability"]:
                # Only convert if it has the .to_dict method (is a DataFrame)
                if hasattr(results_to_save["stability"]["stability_df"], "to_dict"):
                    results_to_save["stability"]["stability_df"] = results_to_save["stability"]["stability_df"].to_dict(
                        "records"
                    )

        with open(filepath, "w") as f:
            json.dump(results_to_save, f, indent=2)
        print("✅ Results saved successfully!")


if __name__ == "__main__":
    import os

    from dotenv import load_dotenv
    from sqlalchemy import create_engine

    load_dotenv()
    DATABASE_URL = os.getenv("DATABASE_URL")
    engine = create_engine(DATABASE_URL)

    # Initialize and run
    selector = FeatureSelector(engine)
    results = selector.run_full_pipeline(run_stability=True, stability_threshold=0.4)

    # This creates the file that train.ipynb is begging for
    selector.save_results(results, "feature_selection_results.json")

NBA FEATURE SELECTION PIPELINE

📊 Season Configuration:
  Training:   22009, 22010, 22011, 22012, 22013, 22014, 22015, 22016, 22017, 22018 (10 seasons)
  Validation: 22021 (1 season)
  Test:       22022, 22023, 22024 (3 seasons)
  Excluded:   22019, 22020 (COVID impact)

🔄 Loading data from database...
  ✈️  Calculating travel logistics...
  📊 Fetching advanced 4 Factors (League Quality)...
  📊 Calculating Strength of Schedule (SOS)...
  ✓ Created ML dataset: 14,716 rows

📈 Data Split Summary:
  Training:   11,264 games
  Validation: 1,152 games
  Test:       2,300 games
  Total:      14,716 games

FEATURE SELECTION

📊 Starting with 479 features

🔍 Step 1: Variance filtering...
  ✓ All features have sufficient variance

🔍 Step 2: ANOVA F-test...
  ✓ Computed F-statistics for 479 features

🔍 Step 3: Mutual Information...
  ✓ Computed MI scores for 479 features

🔍 Step 4: Rank-based fusion...
  ✓ Combined rankings using reciprocal rank fusion

🔍 Step 5: Correlation pruning (threshold=0.8